
---

**1. What is a Databricks Workflow?**

> *A Databricks Workflow is used to orchestrate and schedule data processing jobs in Databricks. We can create multiple tasks, define dependencies between them, run tasks sequentially or in parallel, configure retries and monitor the complete pipeline from a single job.*

**Example flow:**

Ingestion
   ↓
Bronze
   ↓
Silver
   ↓
Gold
   ↓
Validation


---

**2. What is a Job?**

> *A Job is the overall workflow definition that contains one or more tasks. It defines what needs to run, when it should run, which cluster or compute should be used, dependencies, parameters, retries and notifications.*

**Structure:**

Job
 ├── Task 1
 ├── Task 2
 ├── Task 3
 └── Task 4


---

**3. What is a Task?**

> *A task is an individual unit of work inside a Databricks Job. For example, a notebook that loads Bronze data or a SQL task that performs a transformation can be individual tasks.*

---

**4. What types of tasks have you used?**

You can say:
> *I've mainly worked with notebook tasks, Python tasks and SQL tasks. Notebook tasks are useful when the transformation logic is written in Databricks notebooks, Python tasks are useful for reusable Python code, and SQL tasks are useful for SQL-based transformations and validations.*

**Other supported task types include:**
- Notebook
- Python
- SQL
- dbt
- JAR
- Pipeline/Lakeflow-related tasks
- Run Job

---

**5. Notebook Task vs Python Task vs SQL Task**

| Task           | Typical Use                         |
|----------------|-------------------------------------|
| Notebook task  | PySpark/SQL notebook-based processing|
| Python task    | Standalone/reusable Python logic     |
| SQL task       | SQL transformations/queries          |
| Pipeline task  | Lakeflow pipeline execution          |

**Interview answer:**
> *I use notebook tasks when the processing logic is already implemented in a Databricks notebook. For reusable Python logic, I can use a Python task, and for SQL transformations or validations I use SQL tasks.*

---

**6. How do you create dependencies between tasks?**

Suppose:

Bronze
  ↓
Silver
  ↓
Gold

You define dependencies so that:
- Silver depends on Bronze
- Gold depends on Silver

**Interview answer:**
> *In the workflow UI, I define task dependencies using the 'depends on' configuration. This creates a DAG, so Databricks knows which task should run first.*

**Example:**

Bronze
  ↓
Silver
  ↓
Gold


---

**7. How do you implement parallel tasks?**

Suppose Silver produces three independent datasets:

           Bronze
              ↓
       ┌──────┼──────┐
       ↓      ↓      ↓
   Customer  Order  Product
       │      │      │
       └──────┼──────┘
              ↓
             Gold

**Interview answer:**
> *If tasks don't depend on each other, I don't create unnecessary sequential dependencies. I make them independent tasks with the same upstream dependency, so Databricks can execute them in parallel. This reduces the overall pipeline execution time.*

---

**8. How do you configure retries?**

> *At the task level, I can configure the number of retries and retry behavior. I normally use retries for transient failures such as temporary infrastructure or network issues. I don't rely on retries to solve actual data or code errors.*

**Example:**

Task
 ↓
Failed
 ↓
Retry 1
 ↓
Retry 2
 ↓
Failed permanently


**Important interview point:**
> *Retries should be combined with idempotent processing so that retrying a task doesn't create duplicate data.*

---

**9. What happens when one task fails?**

It depends on the dependency structure. For:

Task 1 → Task 2 → Task 3

If Task 2 fails, Task 3 won't execute because its dependency wasn't successful.

**Interview answer:**
> *If a task fails, its downstream dependent tasks normally don't run. The workflow marks the run as failed. I check the task logs and error details, fix the issue and then rerun the appropriate failed task or workflow.*

---

**10. How do you pass parameters between tasks?**

There are multiple approaches.
- **Job parameters example:**  
  
  environment = prod
  process_date = 2026-08-18
  

- A notebook can access job/task parameters through Databricks parameter mechanisms.

- For notebook workflows, you can also use:
  
  dbutils.widgets.get("process_date")
  

- For passing dynamically generated values between tasks, task values can be used.

---

**11. What are Job Parameters?**

> *Job parameters are input values provided to a workflow run. They allow the same workflow to be reused for different environments, dates or processing conditions.*

**Example:**

environment = prod
process_date = 2026-08-18
source = customer

Instead of hardcoding:

process_date = "2026-08-18"

you pass it as a parameter.

---

**12. What are Task Values?**

This is slightly different from job parameters.

> *Task values allow one task to pass a value generated during execution to another task in the same job.*

**Example:**
- Task 1 calculates:
  
  records_processed = 250000
  
- Stores as a task value.

**Conceptually:**

Task 1
   ↓
records_processed = 250000
   ↓
Task 2


**Distinction:**
- Job parameter = input to the workflow
- Task value = output from one task consumed by another task

---

**13. How do you schedule a workflow?**

> *I can configure a schedule directly in the Databricks Job settings, such as hourly, daily or using a cron expression. I can also trigger the workflow on demand or through an external orchestration system.*

**Example:**

Every day at 2 AM
    ↓
Databricks Workflow
    ↓
Bronze → Silver → Gold


---

**14. How do you trigger a workflow from another system?**

> *A Databricks Job can be triggered through APIs or external orchestration tools. For example, an external scheduler can call the Databricks Jobs API to start a job and pass the required parameters.*

**If asked about Azure:**
> *If an enterprise orchestration layer is already being used, such as Azure Data Factory, it can trigger a Databricks job and pass parameters to it.*

**For pure Databricks:**
> *Within Databricks itself, I would use Databricks Workflows and Jobs, and for external triggering I can use the Databricks Jobs API.*

---

**15. How do you monitor workflow runs?**

> *I use the Databricks Jobs UI to monitor the overall workflow and individual task status. For failures, I check the task logs and Spark UI. I also monitor execution time, retries and cluster utilization. For production workflows, I configure notifications for failures.*

**You can explain:**

Job Run
  ↓
Task status
  ↓
Logs
  ↓
Spark UI
  ↓
Error / Performance analysis


---

**16. How do you handle failed workflows?**

> *First I identify the failed task and check the error logs. I determine whether it's a code issue, data issue, permission issue or infrastructure issue. For transient failures, retries can handle the problem. For actual failures, I fix the root cause and rerun the failed task or workflow. I also make sure the pipeline is idempotent so rerunning doesn't create duplicate data.*

---

**17. Job Cluster vs All-Purpose Cluster**

Very commonly asked.

- **All-purpose cluster:**  
  > *An all-purpose cluster is generally used interactively for development, testing and notebook exploration. It can be shared by users.*

- **Job cluster:**  
  > *A job cluster is created specifically for a job run and can be terminated after the job finishes. It's generally better suited for production batch workloads.*

**Simple:**

All-purpose      → Development / Interactive
Job cluster      → Production Jobs


---

**18. Why would you use a Job Cluster?**

> *For production workloads, I prefer job clusters because they provide isolated compute for the job and can automatically terminate after completion. This improves resource utilization and helps control costs.*

Also:
> *It reduces the risk of one user's interactive workload affecting another production workload.*

---

**19. How do you control cluster costs?**

A strong answer:
> *I use job clusters for scheduled production workloads so they're not running continuously. I select an appropriate worker size instead of overprovisioning, enable autoscaling when appropriate, use auto-termination for interactive clusters and monitor job execution time and resource utilization.*

You can also mention:
- Right-size driver/workers
- Avoid unnecessarily large clusters
- Avoid keeping all-purpose clusters running
- Use serverless where appropriate for supported workloads
- Optimize Spark jobs to reduce compute time

---

**20. How do you implement Dev/QA/Prod configurations?**

Don't hardcode environment-specific values inside notebooks.  
Instead:

Dev
 ↓
QA
 ↓
Prod

Use parameters/configuration for:
- catalog
- schema
- storage location
- source
- target
- environment

**Example:**

Dev  → dev_catalog
QA   → qa_catalog
Prod → prod_catalog


**Interview answer:**
> *I keep the code reusable and parameterize environment-specific values such as catalog, schema and processing date. Then the same workflow can run in Dev, QA and Prod with different configurations. I also use separate permissions and resources for each environment.*

---

**🔥 VERY LIKELY SCENARIO**

**Interviewer:**
> *Your pipeline has 5 notebooks. Notebook 3 fails. You don't want to rerun notebooks 1 and 2. How would you design the workflow?*

This is a very good 3.6-year-level question.

**Answer:**
> *I would design each notebook as a separate task in the Databricks Workflow and define dependencies between them.*

**For example:**

Notebook 1
    ↓
Notebook 2
    ↓
Notebook 3
    ↓
Notebook 4
    ↓
Notebook 5

If Notebook 3 fails:

Notebook 1 → SUCCESS
Notebook 2 → SUCCESS
Notebook 3 → FAILED
Notebook 4 → SKIPPED
Notebook 5 → SKIPPED

Then:
> *I would troubleshoot Notebook 3, fix the issue and rerun from Notebook 3 rather than unnecessarily rerunning Notebook 1 and 2. The important part is that Notebook 3 and downstream processing should be idempotent, so retrying or rerunning doesn't duplicate data.*

**🔥 Interviewer may ask: "How do you make Notebook 3 idempotent?"**

Say:
> *I avoid blindly appending the same data every time. For incremental processing, I use checkpoints where applicable and Delta MERGE based on a business key. This ensures that if the task is rerun, existing records are updated rather than duplicated.*

**Example:**

Notebook 3
    ↓
Read incremental data
    ↓
Transform
    ↓
MERGE INTO Silver
    ↓
Success

If it fails and runs again:

Same records
     ↓
MERGE
     ↓
No duplicates


---

**⭐ A slightly more realistic workflow**

If the interviewer wants to see that you understand parallelism, don't always show a straight 1 → 2 → 3 → 4 → 5 pipeline.

**For example:**

                  Ingestion
                      ↓
                    Bronze
                      ↓
               ┌──────┼──────┐
               ↓      ↓      ↓
           Customer  Order  Product
               │      │      │
               └──────┼──────┘
                      ↓
                    Silver
                      ↓
                    Gold
                      ↓
                  Validation

You can say:
> *Customer, Order and Product transformations are independent, so I can run them in parallel. Once all three finish successfully, the Silver or Gold task can start.*

That shows you understand DAG design, not just sequential notebook execution.

---

**🔥 5 Workflows concepts you MUST remember**

For your interview, make these very clear:
1. **Job**  
    Complete workflow
2. **Task**  
    Individual unit of work
3. **Dependency**  
    Controls execution orders
4. **Retry**  
    Handles transient failures
5. **Idempotency**  
    Allows safe reruns without duplicate/wrong data

**Strongest practical statement:**
> *I design workflows as independent, restartable and idempotent tasks, so if a downstream task fails, I can rerun from the failed point without unnecessarily reprocessing successful upstream tasks.*

---